In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# 1. LOAD DỮ LIỆU
try:
    df = pd.read_csv('../../data/processed/training_data_final (only Gold).csv', parse_dates=['Datetime'], index_col='Datetime')
    # Chỉ lấy cột Gold và chuyển về dạng numpy array
    data = df[['Gold']].values
except FileNotFoundError:
    print("❌ Không tìm thấy file dữ liệu.")
    exit()

In [3]:
# 2. CHUẨN HÓA DỮ LIỆU (SCALING) - BẮT BUỘC VỚI LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

In [4]:
# 3. CHIA TRAIN/TEST
# Time-series không được shuffle. Chia 80% đầu train, 20% sau test.
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

In [5]:
# 4. HÀM TẠO DATASET THEO CỬA SỔ TRƯỢT (SLIDING WINDOW)
def create_dataset(dataset, time_step=60):
    X, y = [], []
    for i in range(len(dataset) - time_step - 1):
        # Lấy 60 giá trước đó làm Input (X)
        X.append(dataset[i:(i + time_step), 0])
        # Lấy giá tiếp theo làm Output (y)
        y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(y)

In [ ]:
# Chọn time_step (nhìn lại bao nhiêu phút quá khứ?)
# Ví dụ: 60 có nghĩa là nhìn lại 60 điểm dữ liệu trước đó (nếu data 1 phút/dòng thì là 60 phút)
time_step = 30

X_train, y_train = create_dataset(train_data, time_step)
X_test, y_test = create_dataset(test_data, time_step)

# RESHAPE DATA CHO LSTM [Samples, Time Steps, Features]
# Features = 1 (vì chỉ có Gold)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")

Kích thước X_train: (75922, 60, 1)
Kích thước X_test: (18935, 60, 1)


In [8]:
# 5. XÂY DỰNG MÔ HÌNH LSTM
model = Sequential()

# Layer 1: LSTM
# return_sequences=True nếu nối tiếp thêm layer LSTM nữa
model.add(LSTM(units=50, return_sequences=True, input_shape=(time_step, 1)))
model.add(Dropout(0.2)) # Tránh Overfitting

# Layer 2: LSTM
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))

# Layer Output: Dự đoán 1 giá trị (Giá Gold)
model.add(Dense(units=1))

# Compile
model.compile(optimizer='adam', loss='mean_squared_error')

c:\Users\minam\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# 6. HUẤN LUYỆN (TRAINING)
# epochs: Số vòng lặp, batch_size: Số lượng mẫu học cùng lúc
print("\n🚀 Bắt đầu Train...")
history = model.fit(X_train, y_train, epochs=20, batch_size=128, validation_data=(X_test, y_test), verbose=1)


🚀 Bắt đầu Train...
Epoch 1/20
2373/2373 ━━━━━━━━━━━━━━━━━━━━ 194s 78ms/step - loss: 0.0047 - val_loss: 2.7555e-05
Epoch 2/20
 156/2373 ━━━━━━━━━━━━━━━━━━━━ 1:54 52ms/step - loss: 0.0023